# Predikcija cene polovnih automobila — EDA i modelovanje

Cilj je predvideti `priceUSD`. Ćelije prikazuju strukturu i kvalitet podataka, karakteristike i poređenje regresionih modela. Pokreni iz osnovnog direktorijuma projekta.

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / 'data' / 'cars.csv').exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from data_cleaning import clean_data
from feature_engineering import add_features
from data_preprocessing import FEATURES
raw = pd.read_csv(ROOT / 'data' / 'cars.csv')
df = add_features(clean_data(raw))
print('Shape:', raw.shape, '->', df.shape)
display(raw.head())

## Struktura i nedostajuće vrednosti

In [ ]:
display(pd.DataFrame({'dtype': raw.dtypes.astype(str), 'missing': raw.isna().sum(), 'missing_pct': (100*raw.isna().mean()).round(2)}))
display(raw.describe(include='all').T)

CSV sadrži 56.244 oglasa i 12 kolona. Ciljna cena nema nedostajuće vrednosti. `volume(cm3)` ima 47 praznih vrednosti, `drive_unit` 1.905, a `segment` 5.291. Ti oglasi ostaju u skupu; nedostajuće vrednosti se obrađuju u pipeline-u.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.histplot(raw['priceUSD'], bins=80, ax=axes[0]); axes[0].set_title('Cena (USD), puni raspon')
sns.histplot(raw['priceUSD'].clip(upper=raw['priceUSD'].quantile(.99)), bins=60, ax=axes[1]); axes[1].set_title('Cena do 99. percentila')
plt.tight_layout()

In [ ]:
display(raw['priceUSD'].describe(percentiles=[.01,.05,.25,.5,.75,.95,.99]).to_frame())
print('Cene <= 0:', (raw['priceUSD'] <= 0).sum(), '| >$100k:', (raw['priceUSD'] > 100000).sum())

Cena je asimetrična: medijana je oko $5.350, prosek oko $7.415, 99. percentil oko $39.729, maksimum $235.235. Pozitivne visoke cene ostaju jer mogu predstavljati stvarne oglase; zato se MAE tumači uz RMSE i primere greške.

## Inženjering karakteristika i pretprocesiranje

In [ ]:
display(df[['year','car_age','mileage_km','mileage_per_year','volume_cm3','engine_volume_liters','is_high_mileage','brand_model']].head())
print('Feature count:', len(FEATURES))

Izvedene su starost vozila, litri zapremine, prosečna kilometraža po godini, indikator velike kilometraže (≥250.000 km) i spoj marke/modela. Numeričke vrednosti se imputiraju medianom i standardizuju; kategorije se popunjavaju najčešćom vrednošću i one-hot enkodiraju. Transformacije se uče samo na trening delu.

## Poređenje modela

In [ ]:
from model_training import train
results = train(str(ROOT / 'data' / 'cars.csv'), str(ROOT / 'models'), str(ROOT / 'reports'))
display(results.style.format({'MAE':'${:,.0f}','MSE':'${:,.0f}','RMSE':'${:,.0f}','R2':'{:.3f}'}))

In [ ]:
preds = pd.read_csv(ROOT / 'reports' / 'sample_predictions.csv')
display(preds.head(10).round(0))
summary = json.load(open(ROOT / 'reports' / 'training_summary.json', encoding='utf-8'))
print('Izabrani model:', summary['best_model'])

MAE pokazuje prosečan iznos greške u dolarima. RMSE više kažnjava velike promašaje, a R² opisuje objašnjeni deo varijacije u test skupu. Finalni model bira se po najmanjem MAE na istom hold-out skupu; za buduće cene preporučuje se i vremenska validacija.